# Model Configuration (`model_config`)

Pydantic models come with a set of default behaviors (e.g., ignoring extra fields, lax type coercion, skipping default validation). You can override these defaults at the model level using the `model_config` attribute and the `ConfigDict` object.

## 1. Handling Extra Fields (`extra`)

When you pass data into a Pydantic model (via dictionaries or JSON), what happens if the data contains keys that are *not* defined as fields in your model?

Pydantic provides three behaviors, controlled by the `extra` configuration parameter.

### A. `ignore` (The Default)

By default, Pydantic silently drops any data that doesn't match a defined field.

```python
from pydantic import BaseModel, ConfigDict

class IgnoreModel(BaseModel):
    # This is the implicit default, but shown here explicitly
    model_config = ConfigDict(extra='ignore') 
    
    field_1: int

# Passing an undefined field ('field_2')
m1 = IgnoreModel(field_1=10, field_2=20)

print(m1.model_dump()) 
# {'field_1': 10} -> 'field_2' is completely gone.

```

### B. `forbid` (Strict APIs)

If you are building a strict REST API, you want to ensure the client is sending *exactly* what you asked for and nothing else. If they send an extra field (perhaps due to a typo like `firt_name`), you want to reject the payload.

```python
from pydantic import ValidationError

class ForbidModel(BaseModel):
    model_config = ConfigDict(extra='forbid')
    field_1: int

try:
    m2 = ForbidModel(field_1=10, field_2=20)
except ValidationError as e:
    print("Caught Error: Extra inputs are not permitted")

```

### C. `allow` (Consuming Noisy Data)

If you are consuming a third-party API that returns a massive JSON object with 100 keys, but you only care about validating 2 of them, you can use `allow`. This validates your defined fields and passes the rest of the data through as generic key-value pairs without validation.

```python
class AllowModel(BaseModel):
    model_config = ConfigDict(extra='allow')
    field_1: int

m3 = AllowModel(field_1=10, field_2="ad-hoc string data")

print(m3.model_dump()) 
# {'field_1': 10, 'field_2': 'ad-hoc string data'}

# You can access the extra fields explicitly
print(m3.model_extra) 
# {'field_2': 'ad-hoc string data'}

```

## 2. Using `ConfigDict` vs. Standard Dictionaries

Because `model_config` is technically just a Python dictionary under the hood, you *could* configure a model by writing `model_config = {"extra": "forbid"}`.

**However, you should always use `ConfigDict**`. `ConfigDict` is a `TypedDict`. This means modern IDEs (like VS Code or PyCharm) and static type checkers (like MyPy) will provide auto-complete and immediately warn you if you misspell a configuration key or provide the wrong data type before you even run your code.

### Interview Preparation: Model Configuration & Extra Fields

<details>
<summary><b>1. You are building a FastAPI endpoint. A user sends a POST request with the JSON `{"username": "admin", "is_superuser": true}`. Your Pydantic model only defines `username`. What happens by default, and what security risk might this pose if you aren't careful?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, Pydantic's `extra` configuration is set to `'ignore'`. It will successfully deserialize the `username` and silently drop the `is_superuser` field. <br><br>
The security risk arises if I am using an ORM (like SQLAlchemy) and I pass the raw incoming payload dictionary directly to a database update function instead of using Pydantic's `model_dump()`. If I bypass Pydantic's serialization, the raw dictionary still contains `is_superuser`, which could lead to a Mass Assignment vulnerability where the user elevates their own privileges. If I use `model_dump()`, the ignored field is safely stripped out.
</details>

<details>
<summary><b>2. How can you prevent the scenario described in Question 1 from even occurring at the API gateway level?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
I would change the model configuration to strictly reject unknown fields by setting `model_config = ConfigDict(extra='forbid')`. <br><br>
When the user sends the unexpected `is_superuser` key, Pydantic will instantly throw a `ValidationError`. FastAPI will catch this and automatically return a 422 Unprocessable Entity response to the client, telling them exactly which field was not permitted, securing the application at the boundary.
</details>

<details>
<summary><b>3. You need to consume a legacy API that returns a JSON object with 200 fields. You only need to validate and extract `id` and `status`, but you must save the entire raw JSON payload to an audit database. How do you design your Pydantic model to handle this efficiently?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It would be a massive waste of time to define all 200 fields in a Pydantic model just to pass them through. <br><br>
Instead, I would define a model with only the two fields I care about (`id` and `status`), and set `model_config = ConfigDict(extra='allow')`. Pydantic will strictly validate the ID and status. It will take the other 198 fields and stash them without validation. I can then extract those remaining fields dynamically using the instance's `model_extra` property, or simply use `model_dump()` to get the combined, validated dictionary to push to my audit database.
</details>

<details>
<summary><b>4. Why do we import and use `ConfigDict` from Pydantic instead of just using a standard Python dictionary like `model_config = {"extra": "forbid"}`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
While a standard dictionary works at runtime, it lacks type safety. `ConfigDict` is actually a Python `TypedDict`. By using it, we get robust developer tooling. <br><br>
My IDE will provide auto-complete for all available configuration options, and static type checkers like MyPy will instantly flag if I misspell a key (like typing `extras` instead of `extra`) or pass an invalid value type, catching bugs during development rather than at runtime.
</details>